In [ ]:
"""
Correlation analysis
"""

In [2]:
# ============================================================
# 0. Imports & Font Configuration
# ------------------------------------------------------------
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
     "pandas==2.2.2", "geopandas==1.0.1"])

import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import time as time
from scipy.stats import pearsonr
import seaborn as sns
from sklearn.linear_model import LinearRegression
import seaborn as sns
os.chdir(os.getcwd())

Tnr_path = os.path.join(os.getcwd(), 'Times New Roman.ttf')
font_manager.fontManager.addfont(Tnr_path)
regular_prop = font_manager.FontProperties(fname=Tnr_path)

italic_path = os.path.join(os.getcwd(), 'Roman Italic.ttf')
font_manager.fontManager.addfont(italic_path)
italic_prop = font_manager.FontProperties(fname=italic_path)

plt.rcParams['font.family']      = 'serif'
plt.rcParams['font.serif']       = [regular_prop.get_name()]
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm']      = regular_prop.get_name()
plt.rcParams['mathtext.it']      = italic_prop.get_name()

In [2]:
# ============================================================
# 1. Correlation Analysis-each indices
# ------------------------------------------------------------


if not os.path.exists('data/plot/correlation'):
    os.makedirs('data/plot/correlation')

midsize_cities = gpd.read_file('data/midsize_cities_index.gpkg')
lin_col = ['MLI', 'SLI']
acc_col = ['MTC', 'CVTC', 'MDR', 'CVDR']

correlation_results = []

for lin_suffix in lin_col:
    for acc_suffix in acc_col:
        if lin_suffix in midsize_cities.columns and acc_suffix in midsize_cities.columns:
            correlation, p_value = pearsonr(midsize_cities[lin_suffix], midsize_cities[acc_suffix])
            correlation_results.append({
                'Linearity Column': lin_suffix,
                'Accessibility Column': acc_suffix,
                'Correlation': correlation,
                'P-Value': p_value
            })
            plt.figure(figsize=(8, 8))
            sns.regplot(x=midsize_cities[lin_suffix], y=midsize_cities[acc_suffix], 
                        scatter_kws={'alpha': 0.5}, line_kws={'color': 'red', 'alpha': 0.7})
            plt.title(f'Correlation: {correlation:.3f}\n P-Value: {p_value:.3f}', fontsize=35)
            #plt.xlabel(None, fontsize=12)
            plt.xlabel(lin_suffix, fontsize=12)
            #plt.ylabel(None, fontsize=12)
            plt.ylabel(acc_suffix, fontsize=12)
            plt.tight_layout()
            plt.savefig(f'data/plot/correlation/correlation__{lin_suffix}_{acc_suffix}.png')
            plt.close()
                
correlation_df = pd.DataFrame(correlation_results)
correlation_df['P-Value'] = correlation_df['P-Value'].apply(lambda x: f'{x:.3f}')
correlation_df.to_csv('data/correlation_results.csv', index=False)

In [5]:
from matplotlib.lines import Line2D

# ============================================================
# 2. Correlation Analysis-plot
# ------------------------------------------------------------

if not os.path.exists('data/plot/correlation'):
    os.makedirs('data/plot/correlation')

# Load midsize cities data
midsize_cities = gpd.read_file('data/midsize_cities_index.gpkg')
midsize_cities = midsize_cities[midsize_cities['SIGUNGU_NM'] != '양산시']

# indeces list
lin_col = ['MLI', 'SLI']
acc_col = ['MTC', 'MDR', 'CVTC', 'CVDR']
row_labels = ['MLI', 'SLI']
col_labels = ['MTC', 'MDR', 'CVTC', 'CVDR']

# figure, axes preparation
n_rows, n_cols = len(lin_col), len(acc_col)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))

# Create a grid of scatter plots with regression lines
for i, lin_suffix in enumerate(lin_col):
    for j, acc_suffix in enumerate(acc_col):
        ax = axes[i, j]
        if lin_suffix in midsize_cities.columns and acc_suffix in midsize_cities.columns:
            data = midsize_cities[[lin_suffix, acc_suffix]].dropna()
            x = data[lin_suffix].values
            y = data[acc_suffix].values
        

            # 1) scatter
            sns.scatterplot(x=x, y=y, ax=ax,
                            alpha=0.5, edgecolor='black', linewidth=0.5, s=50, color='gray')

            # 2) regression line
            sns.regplot(x=x, y=y, ax=ax,
                        scatter=False, color='black', line_kws={'alpha': 0.3})

            
            ax.set_xlabel(f'${lin_suffix}$', fontsize=20)
            ax.set_ylabel(f'${acc_suffix}$', fontsize=20)
            
            if pearsonr(x, y)[1] < 0.01 :
                ax.text(0.05, 0.95, f'Corr: {pearsonr(x, y)[0]:.2f}** \n p = {pearsonr(x, y)[1]:.3f}', 
                        transform=ax.transAxes, fontsize=18, ha='left', va='top', color='red')
            elif pearsonr(x, y)[1] < 0.05:
                ax.text(0.05, 0.95, f'Corr: {pearsonr(x, y)[0]:.2f}* \n p = {pearsonr(x, y)[1]:.3f}', 
                        transform=ax.transAxes, fontsize=18, ha='left', va='top', color='red')
            else:
                ax.text(0.05, 0.95, f'Corr: {pearsonr(x, y)[0]:.2f} \n p = {pearsonr(x, y)[1]:.3f}', 
                        transform=ax.transAxes, fontsize=18, ha='left', va='top', color='black')
            
        else:
            ax.set_visible(False)
            ax.text(0.5, 0.5, 'No Data', horizontalalignment='center', verticalalignment='center',
                    fontsize=12, color='gray')

        

# Set titles and labels
plt.tight_layout()
fig.subplots_adjust(left=0.12, 
                    top=0.88,  
                    wspace=0.4,
                    hspace=0.4)


# add label
fig.text(
    0.075,
    -0.02,
    '* p < 0.05    ** p < 0.01', 
    ha='left', 
    fontsize=18
)

# Create custom legend handles for the scatter points and regression line
scatter_handle = Line2D([0], [0], marker='o', color='gray', markersize=8, 
                          markerfacecolor='gray', linestyle='None', markeredgecolor='black', alpha=0.5)
regression_handle = Line2D([0], [0], color='black', lw=2, alpha=0.3)

fig.legend([scatter_handle, regression_handle], ['Data Points', 'Regression Line'], 
           loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=2, fontsize=18, frameon=False)

# save figure
plt.savefig('data/plot/paper/Figure 8.png', dpi=1000, bbox_inches='tight')
plt.savefig('data/plot/correlation/all_correlation.png')
plt.close()

In [4]:
# ============================================================
# *. analysis summary
# ------------------------------------------------------------


midsize_cities_csv = gpd.read_file('data/midsize_cities_index.gpkg')
midsize_cities_csv.drop(columns=['geometry'], inplace=True)

midsize_cities_allstats = midsize_cities_csv[['SIGUNGU_NM_EN', 'pop', 'area', 'nodes', 'links', 'links_length',
                                              'MLI', 'SLI', 'SKI', 'MTC', 'MDR', 'CVDR', 'CVTC']].copy()
midsize_cities_allstats['pop'] = (midsize_cities_allstats['pop'] / 1000).astype(int).apply(lambda x: f"{x:,}")
midsize_cities_allstats['area'] = midsize_cities_allstats['area'].astype(int).apply(lambda x: f"{x:,}")
midsize_cities_allstats['nodes'] = midsize_cities_allstats['nodes'].astype(int).apply(lambda x: f"{x:,}")
midsize_cities_allstats['links'] = midsize_cities_allstats['links'].astype(int).apply(lambda x: f"{x:,}")
midsize_cities_allstats['links_length'] = (midsize_cities_allstats['links_length'] / 1000).astype(int).apply(lambda x: f"{x:,}")
midsize_cities_allstats['MLI'] = midsize_cities_allstats['MLI'].apply(lambda x: f"{x:.2f}")
midsize_cities_allstats['SLI'] = midsize_cities_allstats['SLI'].apply(lambda x: f"{x:.4f}")
midsize_cities_allstats['SKI'] = midsize_cities_allstats['SKI'].astype(int).apply(lambda x: f"{x:,}")
midsize_cities_allstats['MTC'] = midsize_cities_allstats['MTC'].astype(int).apply(lambda x: f"{x:,}")
midsize_cities_allstats['MDR'] = midsize_cities_allstats['MDR'].apply(lambda x: f"{x:.2f}")
midsize_cities_allstats['CVDR'] = midsize_cities_allstats['CVDR'].apply(lambda x: f"{x:.2f}")
midsize_cities_allstats['CVTC'] = midsize_cities_allstats['CVTC'].apply(lambda x: f"{x:.2f}")
midsize_cities_allstats.to_csv('data/midsize_cities_allstats.csv', index=False)

midsize_cities_basicstats = midsize_cities_csv[['SIGUNGU_NM_EN', 'pop', 'area', 'nodes', 'links', 'links_length']].copy()
midsize_cities_basicstats['pop'] = (midsize_cities_basicstats['pop'] / 1000).astype(int).apply(lambda x: f"{x:,}")
midsize_cities_basicstats['area'] = midsize_cities_basicstats['area'].astype(int).apply(lambda x: f"{x:,}")
midsize_cities_basicstats['nodes'] = midsize_cities_basicstats['nodes'].astype(int).apply(lambda x: f"{x:,}")
midsize_cities_basicstats['links'] = midsize_cities_basicstats['links'].astype(int).apply(lambda x: f"{x:,}")
midsize_cities_basicstats['links_length'] = (midsize_cities_basicstats['links_length'] / 1000).astype(int).apply(lambda x: f"{x:,}")
midsize_cities_basicstats.to_csv('data/midsize_cities_basicstats.csv', index=False)

midsize_cities_anlysisstats = midsize_cities_csv[['SIGUNGU_NM_EN', 'MLI', 'SLI', 'SKI', 'MTC', 'MDR', 'CVDR', 'CVTC']].copy()
midsize_cities_anlysisstats['MLI'] = midsize_cities_anlysisstats['MLI'].apply(lambda x: f"{x:.2f}")
midsize_cities_anlysisstats['SLI'] = midsize_cities_anlysisstats['SLI'].apply(lambda x: f"{x:.4f}")
midsize_cities_anlysisstats['SKI'] = midsize_cities_anlysisstats['SKI'].astype(int).apply(lambda x: f"{x:,}")
midsize_cities_anlysisstats['MTC'] = midsize_cities_anlysisstats['MTC'].astype(int).apply(lambda x: f"{x:,}")
midsize_cities_anlysisstats['MDR'] = midsize_cities_anlysisstats['MDR'].apply(lambda x: f"{x:.2f}")
midsize_cities_anlysisstats['CVDR'] = midsize_cities_anlysisstats['CVDR'].apply(lambda x: f"{x:.2f}")
midsize_cities_anlysisstats['CVTC'] = midsize_cities_anlysisstats['CVTC'].apply(lambda x: f"{x:.2f}")
midsize_cities_anlysisstats.to_csv('data/midsize_cities_anlysisstats.csv', index=False)